### Performing LOO-PIT on Field-Level MCMCs with Fixed Initial Conditions

This code book performs LOO-PIT analysis on an MCMC chain provided by Hugo Simon and Arnaud Di Mattia (University of Paris-Saclay). 
The chains are Field-Level inference chains of galaxy density counts over a 32 x 32 x 32 voxel space.
These chains specifically have their initial conditions fixed and only samples over the cosmological parameters. 

First we load the necessary packages

In [ ]:
Pkg.instantiate()
Pkg.resolve()

In [ ]:
# Pkg.add("Turing")
# Pkg.add("Capse")
# Pkg.add("NPZ")
# Pkg.add(url="https://github.com/JuliaCosmologicalLikelihoods/PlanckLite.jl")
# Pkg.add("BenchmarkTools")
# Pkg.add("Plots")
# Pkg.add("Optim")
# Pkg.add("ForwardDiff")
# Pkg.add("LinearAlgebra")
# Pkg.add("Pathfinder")
# Pkg.add("MicroCanonicalHMC")
# Pkg.add("Transducers")
# Pkg.add("StatsPlots")
# Pkg.add("PairPlots")
# Pkg.add("LaTeXStrings")
# Pkg.add("MCMCChains")
# Pkg.add("PSIS")
# Pkg.add("ArviZ")
# Pkg.add("ArviZPythonPlots")
# Pkg.add("IrrationalConstants")
# Pkg.add("DimensionalData")
# Pkg.add("Statistics")

In [ ]:
# using Turing
# using Capse
using NPZ
# # using PlanckLite
# using BenchmarkTools
# using Plots
# using Optim
# using ForwardDiff
# using Pathfinder
# #using MicroCanonicalHMC
# #using Transducers
# using StatsPlots
# #using PairPlots
# #using LaTeXStrings
# #using CairoMakie
# using MCMCChains
using PSIS
# include("utils.jl");
using ArviZ
using ArviZPythonPlots
using LinearAlgebra
using IrrationalConstants
using DimensionalData
using Statistics
using Plots
using Measures
using HypothesisTests

### PSIS

We begin by running Pareto Smoothed Importance Sampling (PSIS) on the full voxels, and storing the results.

The chains provided are a single file, which is loaded below. 

In [ ]:
f = npzread("/Users/ethansmith/Desktop/Waterloo/Masters/LOO-PIT/loop_it_in_the_field/data_hugo/DifferentVoxelSize/lpt_32_fNL0_self_ov1.0_L500/logdf_mesh.npz")
g = npzread("/Users/ethansmith/Desktop/Waterloo/Masters/LOO-PIT/loop_it_in_the_field/data_hugo/DifferentVoxelSize/lpt_32_fNL0_self_ov1.0_L1000/logdf_mesh.npz")
h = npzread("/Users/ethansmith/Desktop/Waterloo/Masters/LOO-PIT/loop_it_in_the_field/data_hugo/DifferentVoxelSize/lpt_32_fNL0_self_ov1.0_L2000/logdf_mesh.npz")

For the process of PSIS, we need the logpdf_mesh field, which are the log probabilities of each voxel that the chain has determined by fitting the model to the full data set. That is, the logpdf_mesh represent $p(y | \theta)$

In [ ]:
f["logpdf_mesh"]

Each of the 32 x 32 x 32 voxels has 4 x 512 delta logdensities from the chain corresponding to that voxel.

Below, we create an array of vectors called logpdf_mesh to store the data from the file. Each of the 32 x 32 x 32 elements has a vector corresponding to a single voxel, and the vector holds the log probability values.

In [ ]:
logpdf_mesh_500 = Array{Vector{Float64}}(undef, 32, 32, 32)
logpdf_mesh_1000 = Array{Vector{Float64}}(undef, 32, 32, 32)
logpdf_mesh_2000 = Array{Vector{Float64}}(undef, 32, 32, 32)

for i in 1:32, j in 1:32, k in 1:32
    logpdf_mesh_500[i,j,k] = []
    logpdf_mesh_1000[i,j,k] = []
    logpdf_mesh_2000[i,j,k] = []
end

for i in 1:32, j in 1:32, k in 1:32
    append!(logpdf_mesh_500[i,j,k], vec(f["logpdf_mesh"][:,:,i,j,k]))
    append!(logpdf_mesh_1000[i,j,k], vec(g["logpdf_mesh"][:,:,i,j,k]))
    append!(logpdf_mesh_2000[i,j,k], vec(h["logpdf_mesh"][:,:,i,j,k]))
end

Thus, log_pdf is a data object representing the 32 x 32 x 32 voxels, with each element holding 2048 log probability values collected from the file.

In [ ]:
logpdf_mesh_500

In [ ]:
logpdf_mesh_500[3,16,4]

Next, we perform Pareto Smoothed Importance Sampling (PSIS) on each voxel's vector of log pdf probabilities.

In [ ]:
# psis_result_500 = Array{PSISResult{Float64, Vector{Float64}, Int64, Int64, PSIS.GeneralizedPareto{Float64}}}(undef, 32, 32, 32);
# psis_result_1000 = Array{PSISResult{Float64, Vector{Float64}, Int64, Int64, PSIS.GeneralizedPareto{Float64}}}(undef, 32, 32, 32);
# psis_result_2000 = Array{PSISResult{Float64, Vector{Float64}, Int64, Int64, PSIS.GeneralizedPareto{Float64}}}(undef, 32, 32, 32);

# for i in 1:32, j in 1:32, k in 1:32
#     psis_result_500[i, j, k] = psis(-logpdf_mesh_500[i,j,k])
#     psis_result_1000[i, j, k] = psis(-logpdf_mesh_1000[i,j,k])
#     psis_result_2000[i, j, k] = psis(-logpdf_mesh_2000[i,j,k])
# end

In [ ]:
using Logging

psis_result_500 = Array{PSISResult{Float64, Vector{Float64}, Int64, Int64, PSIS.GeneralizedPareto{Float64}}}(undef, 32, 32, 32)
psis_result_1000 = similar(psis_result_500)
psis_result_2000 = similar(psis_result_500)

with_logger(NullLogger()) do
    for i in 1:32, j in 1:32, k in 1:32
        psis_result_500[i, j, k] =
            psis(-logpdf_mesh_500[i, j, k])

        psis_result_1000[i, j, k] =
            psis(-logpdf_mesh_1000[i, j, k])

        psis_result_2000[i, j, k] =
            psis(-logpdf_mesh_2000[i, j, k])
    end
end

println("PSIS calculations complete")

The importance weights and the pareto shape values are stored in their own 32 x 32 x 32 arrays so that the PSIS weights and the pareto shape value can be recalled easily for each voxel if needed.

In [ ]:
psis_weights_500 = Array{Vector{Float64}}(undef, 32, 32, 32)
psis_weights_1000 = Array{Vector{Float64}}(undef, 32, 32, 32)
psis_weights_2000 = Array{Vector{Float64}}(undef, 32, 32, 32)

for i in 1:32, j in 1:32, k in 1:32
    psis_weights_500[i,j,k] = psis_result_500[i,j,k].weights
    psis_weights_1000[i,j,k] = psis_result_1000[i,j,k].weights
    psis_weights_2000[i,j,k] = psis_result_2000[i,j,k].weights
end

In [ ]:
println("HELLO")

In [ ]:
pareto_shapes_500 = []
pareto_shapes_1000 = []
pareto_shapes_2000 = []

for i in 1:32, j in 1:32, k in 1:32
    append!(pareto_shapes_500, psis_result_500[i,j,k].pareto_shape)
    append!(pareto_shapes_1000, psis_result_1000[i,j,k].pareto_shape)
    append!(pareto_shapes_2000, psis_result_2000[i,j,k].pareto_shape)
end

In [ ]:
println("HELLO")

In [ ]:
psis_plot_500 = Plots.scatter(pareto_shapes_500; marker =:+, markersize=6, legend = false, linewidth=2)
hline!([0.5,0.7], linestyle=:dot, color =:black)
hline!([1.0], linestyle=:solid, color=:black)

psis_plot_1000 = Plots.scatter(pareto_shapes_1000; marker =:+, markersize=6, legend = false, linewidth=2)
hline!([0.5,0.7], linestyle=:dot, color =:black)
hline!([1.0], linestyle=:solid, color=:black)

psis_plot_2000 = Plots.scatter(pareto_shapes_2000; marker =:+, markersize=6, legend = false, linewidth=2)
hline!([0.5,0.7], linestyle=:dot, color =:black)
hline!([1.0], linestyle=:solid, color=:black)

Plots.plot(psis_plot_500, psis_plot_1000, psis_plot_2000, layout = (1, 3))

The plot above is of the Pareto shape parameters from PSIS. Because the shape parameters are all under the threshhold of 0.5, this informs that the importance sampling is well-behaved and reliable for probability reweighting in the Leave One Out approach.

Now that we know the weights are reliable, we can proceed with the LOO-PIT analysis. The PSIS weights inform how to reweight $p(y | \theta)$ determined by the MCMC logpdf_mesh field, into $p(y_{-i} | \theta)$, the LOO probability density. 

First load the logcdf_mesh field from the MCMC chain file. Similar to logpdf_mesh, this field has contains 4 x 512 samples corresponding to the CDF of that particular voxel. It is stored below in an array with each voxel having an array of all CDF samples

In [ ]:
logcdf_mesh_500 = Array{Vector{Float64}}(undef, 32, 32, 32)
logcdf_mesh_1000 = Array{Vector{Float64}}(undef, 32, 32, 32)
logcdf_mesh_2000 = Array{Vector{Float64}}(undef, 32, 32, 32)

for i in 1:32, j in 1:32, k in 1:32
    logcdf_mesh_500[i,j,k] = []
    logcdf_mesh_1000[i,j,k] = []
    logcdf_mesh_2000[i,j,k] = []
end

for i in 1:32, j in 1:32, k in 1:32
    append!(logcdf_mesh_500[i,j,k], vec(f["logcdf_mesh"][:,:,i,j,k]))
    append!(logcdf_mesh_1000[i,j,k], vec(g["logcdf_mesh"][:,:,i,j,k]))
    append!(logcdf_mesh_2000[i,j,k], vec(h["logcdf_mesh"][:,:,i,j,k]))
end

In [ ]:
logcdf_mesh_500

The CDF values are then able to be systemically reweighted by the PSIS weights in order to obtain the LOO-PIT final values.

In [ ]:
loo_pit_500 = Array{Float64}(undef, 32, 32, 32)
loo_pit_1000 = Array{Float64}(undef, 32, 32, 32)
loo_pit_2000 = Array{Float64}(undef, 32, 32, 32)

for i in 1:32
    for j in 1:32
        for k in 1:32
            loo_pit_500[i,j,k] = dot(psis_weights_500[i,j,k], exp.(logcdf_mesh_500[i,j,k]))
            loo_pit_1000[i,j,k] = dot(psis_weights_1000[i,j,k], exp.(logcdf_mesh_1000[i,j,k]))
            loo_pit_2000[i,j,k] = dot(psis_weights_2000[i,j,k], exp.(logcdf_mesh_2000[i,j,k]))
        end
    end
end

Plotting the LOO-PIT values in a histogram, we see the approximately uniform distribution of values as expected from a MCMC sampling based on a strong model.

In [ ]:
loo_pit_histogram_500 = Plots.histogram(vec(loo_pit_500), legend=false, bins=20, normalize=:pdf, 
                                            ylims=(0,1.2), xticks=0:0.2:1.2, yticks=0:0.2:1.2, color = :firebrick2, linecolor = :firebrick2,
                                            title="L = 500 Mpc/h", xaxis="LOO-PIT", yaxis="density")
hline!([1], color=:red, linestyle=:dash)

loo_pit_histogram_1000 = Plots.histogram(vec(loo_pit_1000), legend=false, bins=20, normalize=:pdf, 
                                            ylims=(0,1.2), xticks=0:0.2:1.2, yticks=0:0.2:1.2, color = :darkorange1, linecolor = :darkorange1,
                                            title="L = 1000 Mpc/h", xaxis="LOO-PIT", yaxis="density")
hline!([1], color=:red, linestyle=:dash)

loo_pit_histogram_2000 = Plots.histogram(vec(loo_pit_2000), legend=false, bins=20, normalize=:pdf, 
                                            ylims=(0,1.2), xticks=0:0.2:1.2, yticks=0:0.2:1.2, color = :steelblue, linecolor = :steelblue,
                                            title="L = 2000 Mpc/h", xaxis="LOO-PIT", yaxis="density")
hline!([1], color=:red, linestyle=:dash)

Plots.plot(loo_pit_histogram_500, loo_pit_histogram_1000, loo_pit_histogram_2000, 
            layout=(1,3), size=(1200,500), bottom_margin=10mm, left_margin=10mm)

In [ ]:
using HypothesisTests
using Distributions

# Flatten the 3D LOO-PIT arrays into vectors
pit_500  = vec(loo_pit_500)
pit_1000 = vec(loo_pit_1000)
pit_2000 = vec(loo_pit_2000)

# Define the null distribution
uniform_dist = Uniform(0, 1)

# Perform one-sample KS tests
ks_500  = ExactOneSampleKSTest(pit_500, uniform_dist)
ks_1000 = ExactOneSampleKSTest(pit_1000, uniform_dist)
ks_2000 = ExactOneSampleKSTest(pit_2000, uniform_dist)

# Print results
println("L = 500 Mpc/h")
println(ks_500)
println("p-value = ", pvalue(ks_500))

In [ ]:
println("\nL = 1000 Mpc/h")
println(ks_1000)
println("p-value = ", pvalue(ks_1000))

In [ ]:
println("\nL = 2000 Mpc/h")
println(ks_2000)
println("p-value = ", pvalue(ks_2000))